In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
%matplotlib qt
# --- 1. SYSTEM PARAMETERS ---
V_SUPPLY = 10.0
I_LIMIT = 1.0
R = 1.0
L = 0.5e-3
KE = 0.02
KT = 0.02
TL = 1.8e-3
B = 0.4e-6
J = 1.5e-6
RAD_TO_M = 1 / (2 * np.pi * 7.5 * 100)

TARGET_POS = 1
T_TOTAL = 15.0
DT = 0.001
TIME = np.arange(0, T_TOTAL, DT)
N = len(TIME)

# --- 2. THE SIMULATION FUNCTION ---
def run_simulation(kp, ki, kd):
    # Initialization
    pos_log = np.zeros(N)
    current_log = np.zeros(N)
    voltage_log = np.zeros(N)
    
    curr_pos = 0.0
    curr_vel = 0.0
    curr_current = 0.0
    integral_error = 0.0
    prev_error = 0.0
    
    for k in range(N):
        # --- Controller (Simulating FPGA) ---
        error = TARGET_POS - curr_pos
        d_error = (error - prev_error) / DT
        
        # Anti-Windup Logic
        if abs(curr_current) < I_LIMIT * 0.95:
            integral_error += error * DT
            
        v_out = (kp * error) + (ki * integral_error) + (kd * d_error)
        v_out = np.clip(v_out, -V_SUPPLY, V_SUPPLY)
        
        # --- Motor Physics ---
        # Electrical
        di_dt = (v_out - (curr_current * R) - (curr_vel * KE)) / L
        curr_current += di_dt * DT
        curr_current = np.clip(curr_current, -I_LIMIT, I_LIMIT)
        
        # Mechanical (Friction logic)
        torque_motor = curr_current * KT
        if abs(curr_vel) < 1e-3 and abs(torque_motor) < TL:
            dw_dt = 0
            curr_vel = 0
        else:
            torque_friction = (B * curr_vel) + (TL * np.sign(curr_vel + 1e-9))
            dw_dt = (torque_motor - torque_friction) / J
            curr_vel += dw_dt * DT
            
        curr_pos += curr_vel * RAD_TO_M * DT
        
        # Logging
        pos_log[k] = curr_pos
        current_log[k] = curr_current
        voltage_log[k] = v_out
        prev_error = error
        
    return pos_log, current_log, voltage_log

# --- 3. INTERACTIVE PLOTTING ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8))
plt.subplots_adjust(bottom=0.25) # Make room for sliders

# Initial Plot
p_init, i_init, v_init = run_simulation(40.0, 15.0, 1.5)

line1, = ax1.plot(TIME, p_init, lw=2, color='blue')
ax1.axhline(y=TARGET_POS, color='red', linestyle='--', label='Target')
ax1.set_ylabel('Position (m)')
ax1.set_title('Live Motor PID Tuning')
ax1.grid(True)
ax1.set_ylim(0, TARGET_POS * 1.3)

line2, = ax2.plot(TIME, i_init, color='red')
ax2.axhline(y=I_LIMIT, color='black', linestyle='--', label='1A Limit')
ax2.set_ylabel('Current (A)')
ax2.grid(True)
ax2.set_ylim(-1.2, 1.2)

line3, = ax3.plot(TIME, v_init, color='green')
ax3.set_ylabel('Voltage (V)')
ax3.set_xlabel('Time (s)')
ax3.grid(True)
ax3.set_ylim(-11, 11)

# Slider Axes [left, bottom, width, height]
ax_kp = plt.axes([0.15, 0.15, 0.65, 0.03])
ax_ki = plt.axes([0.15, 0.10, 0.65, 0.03])
ax_kd = plt.axes([0.15, 0.05, 0.65, 0.03])

# Sliders
sld_kp = Slider(ax_kp, 'Kp', 0.0, 100.0, valinit=40.0)
sld_ki = Slider(ax_ki, 'Ki', 0.0, 50.0, valinit=15.0)
sld_kd = Slider(ax_kd, 'Kd', 0.0, 20.0, valinit=1.5)

def update(val):
    kp = sld_kp.val
    ki = sld_ki.val
    kd = sld_kd.val
    
    new_p, new_i, new_v = run_simulation(kp, ki, kd)
    
    line1.set_ydata(new_p)
    line2.set_ydata(new_i)
    line3.set_ydata(new_v)
    
    fig.canvas.draw_idle()

sld_kp.on_changed(update)
sld_ki.on_changed(update)
sld_kd.on_changed(update)

plt.show()